# 🌐 Multilingual GEC Dataset Builder
### Using [ai4bharat/IndicCorpV2](https://huggingface.co/datasets/ai4bharat/IndicCorpV2)

This notebook walks you through building a clean, native-script corpus for any Indic language — useful as a **Grammatical Error Correction (GEC)** base dataset.

**Pipeline steps:**
1. 📡 Stream sentences from IndicCorpV2 (no full download)
2. 🧹 Remove blank / empty lines
3. 🔢 Convert ASCII digits to native script digits
4. 💾 Save intermediate and final files
5. 📊 Verify the line count

---

## Supported Languages

| Language | Split name | Native digits? |
|---|---|---|
| Tamil | `tam_Taml` | ✅ |
| Telugu | `tel_Telu` | ✅ |
| Malayalam | `mal_Mlym` | ✅ |
| Kannada | `kan_Knda` | ✅ |
| Bengali | `ben_Beng` | ✅ |
| Assamese | `asm_Beng` | ✅ |
| Gujarati | `guj_Gujr` | ✅ |
| Punjabi | `pan_Guru` | ✅ |
| Odia | `ory_Orya` | ✅ |
| Hindi | `hin_Deva` | ✅ |
| Marathi | `mar_Deva` | ✅ |
| Maithili | `mai_Deva` | ✅ |
| Urdu | `urd_Arab` | ✅ (Arabic-Indic) |
| Santali | `sat_Olck` | ➖ (Latin numerals kept) |

---
## 📦 Step 0 — Install dependencies

In [ ]:
!pip install datasets tqdm -q

---
## ⚙️ Step 1 — Configuration

> **Only change things in this cell.** Everything else runs automatically.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ✏️  CONFIGURE YOUR RUN HERE
# ─────────────────────────────────────────────────────────────────────────────

LANGUAGE_SPLIT = "tam_Taml"    # ← Change to your target language split
                               #   See the table in the intro cell above

TARGET_LINES   = 10_000       # Number of non-empty lines to collect
                               # e.g. 10_000 / 100_000 / 500_000 / 800_000 / 1_000_000

OUTPUT_DIR     = "output"      # Folder where all output files will be saved

# ─────────────────────────────────────────────────────────────────────────────
print(f"Language  : {LANGUAGE_SPLIT}")
print(f"Target    : {TARGET_LINES:,} lines")
print(f"Output dir: {OUTPUT_DIR}")

---
## 🛠 Step 2 — Load helpers

In [ ]:
from datasets import load_dataset
from tqdm.notebook import tqdm
import os

# ── Native digit maps ──────────────────────────────────────────────────────────
DIGIT_MAPS = {
    "tam_Taml": {"0":"௦","1":"௧","2":"௨","3":"௩","4":"௪","5":"௫","6":"௬","7":"௭","8":"௮","9":"௯"},
    "tel_Telu": {"0":"౦","1":"౧","2":"౨","3":"౩","4":"౪","5":"౫","6":"౬","7":"౭","8":"౮","9":"౯"},
    "mal_Mlym": {"0":"൦","1":"൧","2":"൨","3":"൩","4":"൪","5":"൫","6":"൬","7":"൭","8":"൮","9":"൯"},
    "kan_Knda": {"0":"೦","1":"೧","2":"೨","3":"೩","4":"೪","5":"೫","6":"೬","7":"೭","8":"೮","9":"೯"},
    "ben_Beng": {"0":"০","1":"১","2":"২","3":"৩","4":"৪","5":"৫","6":"৬","7":"৭","8":"৮","9":"৯"},
    "asm_Beng": {"0":"০","1":"১","2":"২","3":"৩","4":"৪","5":"৫","6":"৬","7":"৭","8":"৮","9":"৯"},
    "guj_Gujr": {"0":"૦","1":"૧","2":"૨","3":"૩","4":"૪","5":"૫","6":"૬","7":"૭","8":"૮","9":"૯"},
    "pan_Guru": {"0":"੦","1":"੧","2":"੨","3":"੩","4":"੪","5":"੫","6":"੬","7":"੭","8":"੮","9":"੯"},
    "ory_Orya": {"0":"୦","1":"୧","2":"୨","3":"୩","4":"୪","5":"୫","6":"୬","7":"୭","8":"୮","9":"୯"},
    "hin_Deva": {"0":"०","1":"१","2":"२","3":"३","4":"४","5":"५","6":"६","7":"७","8":"८","9":"९"},
    "mar_Deva": {"0":"०","1":"१","2":"२","3":"३","4":"४","5":"५","6":"६","7":"७","8":"८","9":"९"},
    "mai_Deva": {"0":"०","1":"१","2":"२","3":"३","4":"४","5":"५","6":"६","7":"७","8":"८","9":"९"},
    "urd_Arab": {"0":"٠","1":"١","2":"٢","3":"٣","4":"٤","5":"٥","6":"٦","7":"٧","8":"٨","9":"٩"},
    "sat_Olck": None,  # Santali — no native digit map, keep ASCII
}

def make_output_dir(directory):
    os.makedirs(directory, exist_ok=True)
    print(f"📁 Output directory ready: {os.path.abspath(directory)}")

def build_path(directory, lang, suffix):
    return os.path.join(directory, f"{lang}_{suffix}.txt")

def save_file(lines, path):
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"💾 Saved → {path}  ({len(lines):,} lines, {size_mb:.1f} MB)")

print("✅ Helpers loaded")

---
## 📡 Step 3 — Fetch from IndicCorpV2 (streaming)

No full download required. The dataset is streamed line-by-line until `TARGET_LINES` non-empty sentences are collected.

In [ ]:
make_output_dir(OUTPUT_DIR)

print(f"\n📡 Streaming IndicCorpV2  |  split = '{LANGUAGE_SPLIT}'  |  target = {TARGET_LINES:,}\n")

dataset = load_dataset(
    "ai4bharat/IndicCorpV2",
    "indiccorp_v2",
    split=LANGUAGE_SPLIT,
    streaming=True,
    trust_remote_code=True,
)

sentences = []
with tqdm(total=TARGET_LINES, unit=" lines", desc="⬇  Fetching") as pbar:
    for sample in dataset:
        text = sample.get("text", "").strip()
        if text:
            sentences.append(text)
            pbar.update(1)
        if len(sentences) >= TARGET_LINES:
            break

print(f"\n✅ Fetched {len(sentences):,} non-empty lines")

raw_path = build_path(OUTPUT_DIR, LANGUAGE_SPLIT, "raw")
save_file(sentences, raw_path)

---
## 🧹 Step 4 — Remove blank lines

In [ ]:
cleaned = [s for s in sentences if s.strip()]
removed = len(sentences) - len(cleaned)
print(f"🧹 Removed {removed:,} blank lines  →  {len(cleaned):,} lines kept")

clean_path = build_path(OUTPUT_DIR, LANGUAGE_SPLIT, "clean")
save_file(cleaned, clean_path)

---
## 🔢 Step 5 — Convert ASCII digits to native script

In [ ]:
digit_map = DIGIT_MAPS.get(LANGUAGE_SPLIT)

if digit_map is None:
    print(f"ℹ️  No native digit map for '{LANGUAGE_SPLIT}' — skipping conversion")
    final = cleaned
else:
    print(f"🔢 Converting digits for '{LANGUAGE_SPLIT}' …")
    final = []
    for line in tqdm(cleaned, desc="🔄  Converting"):
        final.append("".join(digit_map.get(ch, ch) for ch in line))
    print("✅ Digit conversion complete")

final_path = build_path(OUTPUT_DIR, LANGUAGE_SPLIT, f"{TARGET_LINES // 1000}k_native")
save_file(final, final_path)

---
## 📊 Step 6 — Verify final line count

In [ ]:
file_size  = os.path.getsize(final_path)
chunk_size = 1024 * 1024 * 64  # 64 MB chunks
total_lines = 0

with open(final_path, "rb") as f, tqdm(
    total=file_size, unit="B", unit_scale=True, desc="📊 Counting"
) as pbar:
    while True:
        chunk = f.read(chunk_size)
        if not chunk:
            break
        total_lines += chunk.count(b"\n")
        pbar.update(len(chunk))

total_lines += 1  # account for last line if no trailing newline
print(f"\n✅ Final file: {total_lines:,} lines  |  {file_size / (1024*1024):.1f} MB")

---
## 🎉 Summary

In [ ]:
print("="*60)
print("  Pipeline complete!")
print("="*60)
print(f"  Language  : {LANGUAGE_SPLIT}")
print(f"  Target    : {TARGET_LINES:,} lines")
print()
print(f"  Raw    → {raw_path}")
print(f"  Clean  → {clean_path}")
print(f"  Final  → {final_path}")
print("="*60)
print()
print("Next steps for GEC dataset creation:")
print("  1. Use the final file as your CLEAN (correct) reference corpus")
print("  2. Introduce synthetic errors using tools like nlpaug or custom rules")
print("  3. Pair (erroneous, correct) sentences for supervised GEC training")

---
## 📝 Notes

**To switch language**, only edit `LANGUAGE_SPLIT` in Step 1 and re-run all cells.

**Digit conversion** replaces ASCII `0–9` with the language's native numeral glyphs wherever they appear in the text. This is important for authentic script representation in GEC training data.

**Memory tip:** For very large targets (≥ 1M lines), streaming keeps RAM usage low since sentences are processed one at a time.

**IndicCorpV2 HuggingFace page:** https://huggingface.co/datasets/ai4bharat/IndicCorpV2